# 01 · Core DataFrame Operations

Core single-table DataFrame skills: imposing types on raw data,
deriving columns, filtering, and aggregating.

Each task has a **CHECK** cell below it — run it to grade your answer. Write your
solution in the cell with the `# TODO`.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb01")
print("Spark", spark.version, "ready")

sales_raw = read_sales_raw(spark)
products = read_dim(spark, "dim_products")
stores = read_dim(spark, "dim_stores")
customers = read_dim(spark, "dim_customers")
print("raw sales rows:", sales_raw.count())
sales_raw.printSchema()

## Task 1: Impose types (safe casting)

The raw sales CSV has **every column as a string**, and `quantity` even contains junk like `"N/A"`. Build a typed DataFrame `sales` from `sales_raw` with `txn_date`→date, `quantity`→int, `unit_price`→double, `discount`→double. Keep all other columns.

Use **safe** casting so bad values become `null` instead of crashing — Spark 4 runs in ANSI mode, so a plain `.cast('int')` on `"N/A"` throws. Use `try_cast` (`F.expr("try_cast(col as int)")`).

In [ ]:
# TODO: build `sales` with proper types using try_cast
sales = sales_raw  # replace me
sales.printSchema()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
types = dict(sales.dtypes)
assert types["txn_date"] == "date", types["txn_date"]
assert types["quantity"] == "int", types["quantity"]
assert types["unit_price"] == "double"
assert types["discount"] == "double"
# safe cast turned non-numeric quantities into nulls (no crash)
assert sales.filter(F.col("quantity").isNull()).count() > 0
print("✅ Task 1 passed")

## Task 2: Derive a revenue column

Add a `revenue` column to `sales`, defined as `quantity × unit_price × (1 − discount)`. Name the result `sales_rev`. Rows with a null quantity should naturally get a null revenue.

In [ ]:
# TODO: add `revenue`
sales_rev = sales
sales_rev.select("txn_id", "quantity", "unit_price", "discount").show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert "revenue" in sales_rev.columns
_ref = sales.withColumn("_r", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")))
_exp = _ref.agg(F.round(F.sum("_r"), 2)).first()[0]
_got = sales_rev.agg(F.round(F.sum("revenue"), 2)).first()[0]
assert _exp is not None and abs(_exp - _got) < 1.0, (_exp, _got)
print("✅ Task 2 passed — total revenue:", _got)

## Task 3: Filter to valid rows

Create `valid_sales` from `sales_rev` keeping only rows with a **positive quantity** AND a **discount within [0, 1]** (this drops negatives, the `N/A` nulls, and out-of-range discounts). Select just: `txn_id`, `txn_date`, `store_id`, `product_id`, `quantity`, `revenue`.

In [ ]:
# TODO
valid_sales = sales_rev

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert set(valid_sales.columns) == {"txn_id", "txn_date", "store_id", "product_id", "quantity", "revenue"}
assert valid_sales.filter(F.col("quantity") <= 0).count() == 0
assert valid_sales.filter(F.col("revenue").isNull()).count() == 0
print("✅ Task 3 passed — valid rows:", valid_sales.count())

## Task 4: Aggregate per store

From `valid_sales`, compute per `store_id`: `total_revenue` (rounded to 2 dp), `total_units` (sum of quantity), and `n_txns` (row count). Order by `total_revenue` descending. Name it `store_summary`.

In [ ]:
# TODO
store_summary = valid_sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert set(store_summary.columns) == {"store_id", "total_revenue", "total_units", "n_txns"}
_top = store_summary.first()
# the mega-store ST001 should top the revenue ranking
assert _top["store_id"] == "ST001", _top["store_id"]
print("✅ Task 4 passed — top store:", _top["store_id"])

## Task 5: Monthly revenue trend

Add a `year_month` column (formatted `yyyy-MM`) to `valid_sales` and compute monthly total revenue as `monthly_rev` with columns `year_month`, `revenue` (rounded), ordered by `year_month`.

In [ ]:
# TODO
monthly_rev = valid_sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
import re
assert monthly_rev.columns == ["year_month", "revenue"]
assert monthly_rev.count() >= 6
assert all(re.match(r"\d{4}-\d{2}$", r["year_month"]) for r in monthly_rev.collect())
print("✅ Task 5 passed")

---
🎉 Finished! Re-run the notebook top-to-bottom; every CHECK cell should print a ✅.